In [37]:
import torch
from torch import nn
import matplotlib as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import functional as F

In [38]:
# getting data
pd_bank_data_train = pd.read_csv("train.csv")
pd_bank_data_test = pd.read_csv("test.csv")

In [39]:
# printing data
print(pd_bank_data_train.shape)
pd_bank_data_train.head()

(165034, 14)


,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.00,2,1.0,0.0,181449.97,0
1,1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.00,2,1.0,1.0,49503.50,0
2,2,15694510,Hsueh,678,France,Male,40.0,10,0.00,2,1.0,0.0,184866.69,0
3,3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0
4,4,15766172,Chiemenam,716,Spain,Male,33.0,5,0.00,2,1.0,1.0,15068.83,0


In [40]:
# добавляем новые столбцы в test
pd_bank_data_test['BalanceSalaryRatio'] = pd_bank_data_test['Balance'] / (pd_bank_data_test['EstimatedSalary'] + 1)
pd_bank_data_test['CreditScoreAge'] = pd_bank_data_test['CreditScore'] / pd_bank_data_test['Age']
pd_bank_data_test['TenureAge'] = pd_bank_data_test['Tenure'] / pd_bank_data_test['Age']
pd_bank_data_test['IsActiveHighBalance'] = \
((pd_bank_data_test['IsActiveMember'] == 1) & (pd_bank_data_test['Balance'] > pd_bank_data_test['Balance'].median())).astype(int)

# добавляем новые столбцы в train
pd_bank_data_train['BalanceSalaryRatio'] = pd_bank_data_train['Balance'] / (pd_bank_data_train['EstimatedSalary'] + 1)
pd_bank_data_train['CreditScoreAge'] = pd_bank_data_train['CreditScore'] / pd_bank_data_train['Age']
pd_bank_data_train['TenureAge'] = pd_bank_data_train['Tenure'] / pd_bank_data_train['Age']
pd_bank_data_train['IsActiveHighBalance'] = \
((pd_bank_data_train['IsActiveMember'] == 1) & (pd_bank_data_train['Balance'] > pd_bank_data_train['Balance'].median())).astype(int)

# меняем место столбца Exited
pd_bank_data_train = pd_bank_data_train[[c for c in pd_bank_data_train.columns if c != 'Exited'] + ['Exited']]

# проверяем размеры
print(f"Доля тех, кто ушел из банка {(pd_bank_data_train["Exited"].sum()/pd_bank_data_train.shape[0]):.4f}")
print("Train shape: \t", pd_bank_data_train.shape)
print("Test shape: \t", pd_bank_data_test.shape)
pd_bank_data_train.head()

Доля тех, кто ушел из банка 0.2116
Train shape: 	 (165034, 18)
Test shape: 	 (110023, 17)


,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,BalanceSalaryRatio,CreditScoreAge,TenureAge,IsActiveHighBalance,Exited
0,0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.00,2,1.0,0.0,181449.97,0.000000,20.242424,0.090909,0,0
1,1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.00,2,1.0,1.0,49503.50,0.000000,19.000000,0.030303,0,0
2,2,15694510,Hsueh,678,France,Male,40.0,10,0.00,2,1.0,0.0,184866.69,0.000000,16.950000,0.250000,0,0
3,3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,1.760634,17.088235,0.058824,1,0
4,4,15766172,Chiemenam,716,Spain,Male,33.0,5,0.00,2,1.0,1.0,15068.83,0.000000,21.696970,0.151515,0,0


In [41]:
# проверка на наличие пропуском. Есть есть - заполняем дальше в нашем классе
for col in pd_bank_data_train.columns:
    print(f"{col:<20} | {pd_bank_data_train[col].isna().sum()} NULL objects | DATA TYPE: {pd_bank_data_train[col].dtypes}")

id                   | 0 NULL objects | DATA TYPE: int64
CustomerId           | 0 NULL objects | DATA TYPE: int64
Surname              | 0 NULL objects | DATA TYPE: object
CreditScore          | 0 NULL objects | DATA TYPE: int64
Geography            | 0 NULL objects | DATA TYPE: object
Gender               | 0 NULL objects | DATA TYPE: object
Age                  | 0 NULL objects | DATA TYPE: float64
Tenure               | 0 NULL objects | DATA TYPE: int64
Balance              | 0 NULL objects | DATA TYPE: float64
NumOfProducts        | 0 NULL objects | DATA TYPE: int64
HasCrCard            | 0 NULL objects | DATA TYPE: float64
IsActiveMember       | 0 NULL objects | DATA TYPE: float64
EstimatedSalary      | 0 NULL objects | DATA TYPE: float64
BalanceSalaryRatio   | 0 NULL objects | DATA TYPE: float64
CreditScoreAge       | 0 NULL objects | DATA TYPE: float64
TenureAge            | 0 NULL objects | DATA TYPE: float64
IsActiveHighBalance  | 0 NULL objects | DATA TYPE: int64
Exited      

In [42]:
# Что делаем с каждым столбцом
# 0 - удалить
# 1 - не менять
# 2 - отделить по классам
to_do_column = [ 0, #id
                 0, #CustomerId
                 0, #Surname
                 1, #CreditScore
                 2, #Geography
                 2, #Gender
                 1, #Age
                 1, #Tensure
                 1, #Balance
                 1, #NumOfProducts
                 1, #HasCrCard
                 1, #IsActiveMember
                 1, #EstimatedSalary
                 1, #BalanceSalaryRatio
                 1, #CreditScoreAge
                 1, #TenureAge
                 1, #IsActiveHighBalance
                 
]
np_to_do_column = np.array(to_do_column)

In [43]:
# класс обрботки данных
class process_class():
    def __init__(self):
        self.counter_numeric = 0
        self.counter_category = 0
    
    def precessing(self, pandas_data, how_to_process):
        """
        input
        how_to_process - numpy array with proccessing labels [0, 1, 2]
        pandas_data - pandas array train/test array for data processing. Only features with no target column
     
        return:
        np_data_concatenate_numeric - numereic data array after processing
        np_data_concatenate_category - category data array after processing
        header - numpy array for new headers
        """
        header = np.array([])
        np_data_concatenate_numeric = np.zeros((pandas_data.shape[0], 1))
        np_data_concatenate_category = np.zeros((pandas_data.shape[0], 1))
    
        counter_numeric = 0
        counter_category = 0
        
        for col, i in zip(pandas_data.columns, range(how_to_process.shape[0])):
            print(f"i = {i}")
            if (how_to_process[i] == 0):
                # удаляем столбец. Он не нужен. А лучше сказать по другому - мы его не будет вставлять в новый numpy array
                print(f"Column name: {col}")
                print(f"PROCESS TYPE - {how_to_process[i]}")
                print("Column not included \n")        
                continue
        
            elif (how_to_process[i] == 1):
                # копируем из pandas и вставляем в numpy array
                print(f"Column name: {col}")
                print(f"PROCESS TYPE - {how_to_process[i]}")
                counter_numeric += 1 # для проверки
                
                np_add = pandas_data[col].fillna(0).to_numpy()[:, None] # превращаем NaN в нули
                np_data_concatenate_numeric = np.concatenate((np_data_concatenate_numeric, np_add), axis = 1)
                nan_sum = np.isnan(np_add).sum()
                null_num = pandas_data[col].isnull().sum()
                print(f"Null num: {null_num}") # для проверки
                print(f"NaN  num: {nan_sum} \n") # для проверки. Если показывает 0 - все верно
        
                # добавляем названия в header
                header = np.append(header, col)
        
            elif (how_to_process[i] == 2):
                print(f"Column name: {col}")
                print(f"PROCESS TYPE - {how_to_process[i]}")
        
                null_num = pandas_data[col].isnull().sum()
                print(f"Null num: {null_num}") # для проверки
                unique_data_column = pandas_data[col].unique()
                print("Unique data types", unique_data_column.shape[0])
                
                pandas_data[col] = pandas_data[col].fillna("None") # убираем NULL
                null_num = pandas_data[col].isnull().sum()
                print(f"Null num after changed: {null_num}") # для проверки 
                unique_data_column = pandas_data[col].unique()
                print("Unique data types after changed", unique_data_column.shape[0]) # для проверки 
        
                counter_category += unique_data_column.shape[0] # для проверки
                
                # добавляем в наш массив
                np_add = np.zeros((pandas_data.shape[0], unique_data_column.shape[0])) # превращаем NaN в нули
                print("shape", np_add.shape)
                
                for i_new in range(pandas_data.shape[0]):
                    for j_new in range(unique_data_column.shape[0]):
                        if (pandas_data.loc[i_new, col] == unique_data_column[j_new]):
                            np_add[i_new, j_new] = 1 # этот элемент есть
                        else:
                            continue # этого элемента нет - обнуляем
        
                np_data_concatenate_category = np.concatenate((np_data_concatenate_category, np_add), axis = 1)
        
                # добавляем названия в header
                for i in range(unique_data_column.shape[0]):
                    head = str(unique_data_column[i]) + "_" + str(col)
                    header = np.append(header, head)
        
                print("\n")
        
        np_data_concatenate_numeric = np_data_concatenate_numeric[:, :-1] # убираем первый пустой столбец, который добавили в самом начале
        np_data_concatenate_category = np_data_concatenate_category[:, :-1] 

        self.counter_numeric = counter_numeric
        self.counter_category = counter_category
        
        return np_data_concatenate_numeric, np_data_concatenate_category, header

    def concate_display(self, np_data_concatenate_numeric, np_data_concatenate_category, header):
        """
        concatenates numeric and category arrays  
        prints the data after
        """
        np_data_concatenate = np.concatenate((np_data_concatenate_numeric, np_data_concatenate_category), axis = 1)
        
        print("Численные признаки      =", np_data_concatenate_numeric.shape[1], " == ", self.counter_numeric)
        print("Категориальные признаки =", np_data_concatenate_category.shape[1], " == ", self.counter_category)
        print("Размер header           =", header.shape[0])
        print("Количество в header прав=", self.counter_numeric + self.counter_category == header.shape[0])
        print("Размер данных матрицы   =", np_data_concatenate.shape)
        print("Есть ли пропуски        =", np.isnan(np_data_concatenate).any())

        return np_data_concatenate

In [44]:
# обработка данных 
PROC = process_class()
numeric_train, category_train, header_train = PROC.precessing(pd_bank_data_train, np_to_do_column)
processed_data_np = PROC.concate_display(numeric_train, category_train, header_train)

i = 0
Column name: id
PROCESS TYPE - 0
Column not included 

i = 1
Column name: CustomerId
PROCESS TYPE - 0
Column not included 

i = 2
Column name: Surname
PROCESS TYPE - 0
Column not included 

i = 3
Column name: CreditScore
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 4
Column name: Geography
PROCESS TYPE - 2
Null num: 0
Unique data types 3
Null num after changed: 0
Unique data types after changed 3
shape (165034, 3)


i = 5
Column name: Gender
PROCESS TYPE - 2
Null num: 0
Unique data types 2
Null num after changed: 0
Unique data types after changed 2
shape (165034, 2)


i = 6
Column name: Age
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 7
Column name: Tenure
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 8
Column name: Balance
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 9
Column name: NumOfProducts
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 10
Column name: HasCrCard
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 11
Column name: IsActiveMember
PROCESS TYPE - 1
Null num: 0

In [45]:
# берем обработанные данные
x_data = processed_data_np
y_data = pd_bank_data_train["Exited"].to_numpy()

In [46]:
# train test split
x_train_np, x_test_np, y_train_np, y_test_np = train_test_split(x_data, y_data, test_size = 0.25, stratify=y_data, random_state=42, shuffle=True)

In [47]:
# Проверка верного разбиения по классам
print("Доля дефолтов в выборке original", (y_data == 1).sum()/y_data.shape[0]) 
print("Доля дефолтов в выборке train \t", (y_train_np == 1).sum()/y_train_np.shape[0])
print("Доля дефолтов в выборке test \t", (y_test_np == 1).sum()/y_test_np.shape[0])

Доля дефолтов в выборке original 0.21159882206090866
Доля дефолтов в выборке train 	 0.21160169662694406
Доля дефолтов в выборке test 	 0.211590198502145


Теперь будем решать через леса

In [48]:
#preparing pipeling for the RandomForest
Pipeline = sklearn.pipeline.Pipeline
RandomForest = sklearn.ensemble.RandomForestClassifier

# pipleline with Scaler
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("forest", RandomForest(random_state=42))
])

In [49]:
#Grid for RandomForest
param_grid = {
    'forest__n_estimators': [300, 500, 1000],
    'forest__max_depth': [3, 5, 7, 10, None],
    'forest__min_samples_split': [10, 30, 100],
    'forest__min_samples_leaf': [5, 10, 20, 50],
    'forest__max_features': ['sqrt', 0.1, 0.2, 0.3, 0.5], 
    'forest__class_weight': ['balanced']
}

In [50]:
%%time 

# Ищем лучшие параметры для леса
random_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter = 60, cv = 5, scoring='roc_auc', verbose=1, n_jobs=-1)
random_search.fit(x_train_np, y_train_np)

# Лучшие параметры
print(f"Best params for RandomSearch: {random_search.best_params_}")
print(f"Best score for RandomSearch: {random_search.best_score_:.4f}")

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best params for RandomSearch: {'forest__n_estimators': 300, 'forest__min_samples_split': 100, 'forest__min_samples_leaf': 10, 'forest__max_features': 0.5, 'forest__max_depth': 10, 'forest__class_weight': 'balanced'}
Best score for RandomSearch: 0.8874
CPU times: total: 1min 55s
Wall time: 1h 11min 30s


In [51]:
#function for writing data into txt file
def write_txt(name, array_best_param, best_score):
    """
    writing parameters in new txt file
    """
    name_txt = name + ".txt"
    with open(name, 'w') as f:
        f.write(f"ROC AUC score {best_score:.4f}\n")
        f.write("BEST PARAM INFO\n")
        for key in array_best_param.keys():
            f.write(f"{key:<30} | {array_best_param[key]}\n")     

#Запишем сразу
write_txt("randomsearch_1", random_search.best_params_, random_search.best_score_)

In [52]:
# Берем лучшие параметры
best_params = random_search.best_params_
best_RandomForest = RandomForest(
    n_estimators=best_params['forest__n_estimators'],
    max_depth=best_params['forest__max_depth'],
    min_samples_split=best_params['forest__min_samples_split'],
    class_weight=best_params['forest__class_weight'],
    random_state=42
)

# Обучаемся на всей выборке
best_RandomForest.fit(x_train_np, y_train_np)

#predict
loan_predicted_train = best_RandomForest.predict_proba(x_train_np)[:, 1]
loan_predicted_test = best_RandomForest.predict_proba(x_test_np)[:, 1]

#score 
print("ROC AUC TRAIN: \t", roc_auc_score(y_train_np, loan_predicted_train))
print("ROC AUC TEST: \t", roc_auc_score(y_test_np, loan_predicted_test))

ROC AUC TRAIN: 	 0.899098433650877
ROC AUC TEST: 	 0.8858751273029191


Подготовка к отправке данных

In [53]:
# обработка данных 
PROC = process_class()
numeric_test, category_test, header_test = PROC.precessing(pd_bank_data_test, np_to_do_column)
processed_data_np_test = PROC.concate_display(numeric_test, category_test, header_test)

i = 0
Column name: id
PROCESS TYPE - 0
Column not included 

i = 1
Column name: CustomerId
PROCESS TYPE - 0
Column not included 

i = 2
Column name: Surname
PROCESS TYPE - 0
Column not included 

i = 3
Column name: CreditScore
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 4
Column name: Geography
PROCESS TYPE - 2
Null num: 0
Unique data types 3
Null num after changed: 0
Unique data types after changed 3
shape (110023, 3)


i = 5
Column name: Gender
PROCESS TYPE - 2
Null num: 0
Unique data types 2
Null num after changed: 0
Unique data types after changed 2
shape (110023, 2)


i = 6
Column name: Age
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 7
Column name: Tenure
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 8
Column name: Balance
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 9
Column name: NumOfProducts
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 10
Column name: HasCrCard
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 11
Column name: IsActiveMember
PROCESS TYPE - 1
Null num: 0

In [55]:
# predict
loan_predicted_test_send = best_RandomForest.predict_proba(processed_data_np_test)[:, 1]

In [56]:
# Загружаем тестовые данные (нужны только Id)
test_ids = pd_bank_data_test['id']

# Создаём DataFrame для сабмита
submission = pd.DataFrame({
    'id': test_ids,
    'Exited': loan_predicted_test_send
})

# Проверяем формат
submission.head()

,id,Exited
0,165034,0.053451
1,165035,0.904675
2,165036,0.065468
3,165037,0.649105
4,165038,0.430255


In [57]:
# Сохраняем в CSV (без индекса!)
submission.to_csv('submission.csv', index=False)